# Spark Adaptive Query Execution (AQE) Deep Dive

This interactive notebook lets you run PySpark operations step-by-step to observe how the **Spark Engine** processes jobs under the hood when **Adaptive Query Execution (AQE)** is enabled.

By enabling AQE and setting the initial shuffle partitions to `200`, you will inspect the live Spark UI to witness three core AQE optimizations in action:
1. **Dynamic Partition Coalescing**: Reducing reducer tasks dynamically from `200` down to `1` or `2` for small datasets to avoid task-scheduler overhead.
2. **Adaptive Stage Execution (One Job per Stage)**: How Spark executes queries stage-by-stage, submitting new stages as individual Jobs to update execution plans at runtime.
3. **Reusing Exchanges & Skipped Stages**: Observing why certain stages appear as **skipped** in the Spark UI when Spark reuses already computed shuffle outputs.

As you execute each cell, you can explore the live **Spark UI** at **[http://localhost:4040](http://localhost:4040)**.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize Spark Session with AQE enabled
# We set the initial shuffle partitions to 200 (Spark's default) to see dynamic coalescing
spark = SparkSession.builder \
    .appName("SparkEngineDemo-Lab4-AQE") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Active! AQE is ENABLED. Log level set to ERROR.")
print("👉 Spark UI exposes at: http://localhost:4040")

## Step 1: Read Orders Dataset (No Schema Inference)

First, we read the `sample_orders.csv` dataset. We configure `.option("inferSchema", "false")` and `.option("header", "true")`.

### 🔍 Spark UI Observation:
1. Run the cell below.
2. Check the **Jobs** tab in the Spark UI. You will notice that **exactly 1 job (Job 0)** has been created.
3. **Why?** Even though Spark is lazy and type inference is disabled, because we specified `.option("header", "true")`, Spark must eagerly submit 1 job to read the first line of the CSV file to resolve the header column names.

In [ ]:
orders_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("sample_orders.csv")

orders_df.printSchema()
print("\nOrders loaded. Check the Spark UI - exactly 1 job (Job 0) should be created!")

## Step 2: Read Customers Dataset (With Schema Inference)

Next, we read the `sample_customers.csv` dataset in a separate cell, this time setting `.option("inferSchema", "true")` and `.option("header", "true")`.

### 🔍 Spark UI Observation:
1. Run the cell below.
2. Go to the **Jobs** tab in the Spark UI. You will notice that **exactly 2 jobs (Job 1 and Job 2)** have been created.
3. **Why?** Because both `header` and `inferSchema` are set to `true`, Spark must perform two eager read operations: one job to retrieve the column headers, and a second job to scan the rows and infer their data types.

In [ ]:
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("sample_customers.csv")

customers_df.printSchema()
print("\nCustomers loaded. Check the Spark UI - exactly 2 jobs (Job 1 and Job 2) should be created!")

## Step 3: Define Narrow Transformations & Observe Lazy Evaluation

We define a filter and select transformation on our orders dataframe. These are **Narrow Transformations** because data does not move across executors.

### 🔍 Spark UI Observation:
1. Run the cell below.
2. Refresh the Spark UI. No new jobs appear because Spark records the DAG path lazily without initiating computation.

In [ ]:
orders_filtered = orders_df \
    .filter(F.col("Amount") > 150.0) \
    .select("OrderID", "CustomerID", "Amount")

print("Transformation defined! Check Spark UI—no new jobs should appear.")

## Step 4: Trigger Action 1 (Narrow Transformations)

We trigger the `.count()` action on the filtered orders.

### 🔍 Spark UI Observation:
1. Run the cell below.
2. A new **Job 3** will appear in the Spark UI.
3. Click on **Job 3** to view its details.
4. **Observe the Stages**: It contains exactly **1 Stage**.
5. **Observe the DAG**: You will see a straight-line flow containing `FileScan` -> `Filter` -> `Project` (Select) -> `Exchange/HashAggregate` all in one block. 
6. **Why?** Since no data shuffle was required, Spark executed the entire pipeline inside a single Stage.

In [ ]:
count = orders_filtered.count()
print(f"Filtered Orders Count = {count}")
print("\nCheck Job 3 in Spark UI. Observe that this job has exactly 1 Stage and a straight-line DAG.")

## Step 5: Wide Transformation & Dynamic Partition Coalescing (AQE Enabled)

We group the orders by `CustomerID` and sum the `Amount`. `groupBy` is a **Wide Transformation**.

### 🔍 Spark UI Observation:
1. Run the cell below.
2. Go to the **Jobs** tab in the Spark UI. Notice what happens with AQE enabled:
   * **Multiple Jobs Created**: Instead of 1 job with multiple stages, AQE executes the query **stage-by-stage**. You will see that **Job 4** and **Job 5** are triggered sequentially.
   * **Why?** AQE materializes the shuffle stages first (Job 4, Stage A), gathers partition metrics, and replans the next stage. It then submits a separate job (Job 5, Stage B) with an optimized plan.
   * **Dynamic Partition Coalescing**: Click on the second job (the reduce stage). Look at the task list. You will notice that instead of launching `200` tasks (from the `spark.sql.shuffle.partitions` configuration), Spark coalesced the partitions down to **1 task**!
   * **Why?** AQE analyzed the shuffle write files at runtime, saw they were tiny, and merged the 200 small partitions into 1 large partition to avoid task scheduling overhead.

In [ ]:
orders_grouped = orders_df \
    .groupBy("CustomerID") \
    .agg(F.round(F.sum("Amount"), 2).alias("TotalSales"))

# Trigger Action via complete write to no-operation format
orders_grouped.write.mode("overwrite").format("noop").save()
print("Check Jobs 4 & 5 in Spark UI. Observe the stage-by-stage jobs and partition coalescing!")

## Step 6: Complex DAG (Join Transformation - Skipped Stages & Exchange Reuse)

We join the aggregated orders with the customer dataset on `CustomerID`.

### 🔍 Spark UI Observation:
1. Run the cell below.
2. Look at the new jobs in the Spark UI. You will notice that **Job 6** and **Job 7** are triggered.
3. **Observe the Skipped Stages**:
   * Open **Job 7** details. You will notice that one of its parent stages is marked as **skipped**!
   * **Why?** This is AQE's **Exchange Reuse** in action. Since the aggregated orders shuffle write was already computed and materialized during Step 5 (Job 4), AQE detects this and reuses the shuffle files directly. It skips recalculating the orders stage entirely!

In [ ]:
final_report = orders_grouped.join(customers_df, on="CustomerID", how="inner") \
    .select("CustomerID", "CustomerName", "TotalSales", "Country")

# Trigger Action via complete write to no-operation format
final_report.write.mode("overwrite").format("noop").save()
print("Check Jobs 6 & 7 in the Spark UI. Observe the skipped stages indicating shuffle reuse!")

## Step 7: Exploring the SQL / DataFrame Tab in Spark UI

While the **Jobs** tab shows the execution timeline, the **SQL/DataFrame** tab exposes the underlying optimizer plans and execution details. Every time you trigger an action on a DataFrame (like `.show()`, `.write()`, or `.count()`), Spark logs the operation details under the **SQL/DataFrame** tab.

### 🔍 Spark UI Observation:
1. Run the cell below to register temporary SQL views and run a Spark SQL query.
2. Open the **SQL/DataFrame** tab in your browser (`http://localhost:4040/SQL/`).
3. Click on the link for the query you just executed (labeled `save at <ipython-input-...>`).
4. **Inspect the Query Plan Visualization**:
   * **AdaptiveSparkPlan**: Notice a wrapping block called `AdaptiveSparkPlan`. This indicates that AQE is managing the execution.
   * **Coalesced Shuffle**: Observe the Exchange nodes. You will see metrics showing that the shuffle partitions were coalesced dynamically at runtime.
   * **SortMergeJoin** / **BroadcastHashJoin**: Click on the details toggle to inspect plans. You can see how AQE dynamically simplifies the plan based on intermediate data sizes.

In [ ]:
# Register temporary views for SQL operations
orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")

# Execute a SQL Join Aggregation query
sql_df = spark.sql("""
    SELECT 
        c.Country,
        ROUND(SUM(CAST(o.Amount AS DOUBLE)), 2) as TotalRevenue
    FROM orders o
    JOIN customers c ON o.CustomerID = c.CustomerID
    WHERE CAST(o.Amount AS DOUBLE) > 100.0
    GROUP BY c.Country
    ORDER BY TotalRevenue DESC
""")

# Trigger Action via complete write to no-operation format
sql_df.write.mode("overwrite").format("noop").save()
print("SQL/DataFrame query executed. Explore its visualization and physical details in the Spark UI!")

## Step 8: Shutdown Spark Session

In [ ]:
spark.stop()
print("Spark Session stopped successfully.")